<a href="https://colab.research.google.com/github/Akhila-010/GenAI_Tasks/blob/main/ChatBot_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai faiss-cpu PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 9.2 MB/s eta 0:00:00


In [ ]:
from openai import OpenAI
from getpass import getpass

OPENAI_API_KEY = getpass("Enter your OpenAI API key:")
client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving information-brochure.pdf to information-brochure.pdf


In [ ]:
import PyPDF2

def extract_text_from_pdfs(uploaded_files):
    all_text = ""

    for file_name in uploaded_files:
        reader = PyPDF2.PdfReader(file_name)

        for page in reader.pages:
            text = page.extract_text()
            if text:
                all_text += text + "\n"

    return all_text

text = extract_text_from_pdfs(uploaded)
print(text[:1000])

1  INFORMATION BROCHURE  
(Updated on May 4 , 2020)  
(Semester I, 2020 -2021)  
 
 
 
 
 
 
for admission to  
Ph. D., M. Tech., M.S.(R), M. Des. and M. Sc.* Programmes  
 
(For Indian Applicants)  
 
 
 
 
 
INDIAN INSTITUTE OF TECHNOLOGY  DELHI 
HAUZ KHAS, NEW DELHI 110016  
 
 
 
 
 
 
 
 
 
* For M.Sc. (Cognitive Science) & M.Sc.(Economics) only. Admission to other M.Sc. programmes is on the basis of JAM.  

2   
 
 
CONTENTS  
 
 
 
Message to the Applicants from Dean, Academics  3 
Important Dates  5 
Introduction  6 
Credit System  7 
Admission Procedures and Requirements  7 
(I) Ph.D. Programmes  7 
(II) M.Tech. / M.S. (Research) / M. Des./ Programmes  11 
(III) M. Sc. Programmes  (Cognitive Science and Economics)  15 
Reservation of seats  16 
Registration for courses  17 
Hostel accommodation  17 
Fees and payments  17 
(a) Institute dues payable by new students of Ph. D./ M.Tech./ M.S.(R) / M.Des. / M.Sc.  17 
(b) Mess dues payable by  new students  18 
Financial assistance

In [ ]:
def chunk_text(text, size=500, overlap=100):
    chunks = []

    for i in range(0, len(text), size - overlap):
        chunks.append(text[i:i+size])

    return chunks

chunks = chunk_text(text)
print("Total chunks:", len(chunks))

Total chunks: 358


In [ ]:
def get_embeddings(chunks):
    embeddings = []

    for chunk in chunks:
        response = client.embeddings.create(
            model="text-embedding-3-small",
            input=chunk
        )
        embeddings.append(response.data[0].embedding)

    return embeddings

embeddings = get_embeddings(chunks)

In [ ]:
import faiss
import numpy as np

embeddings_np = np.array(embeddings).astype("float32")

index = faiss.IndexFlatL2(len(embeddings_np[0]))
index.add(embeddings_np)

In [ ]:
def search(query, k=5):
    query_embedding = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    ).data[0].embedding

    D, I = index.search(np.array([query_embedding]).astype("float32"), k)

    return [chunks[i] for i in I[0]]

In [ ]:
def ask(query):
    relevant_chunks = search(query)
    context = "\n".join(relevant_chunks)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer ONLY from the given context. If not found, say 'Not in documents'."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion:{query}"}
        ]
    )

    return response.choices[0].message.content

In [ ]:
print(" Multi-PDF College Assistant Ready!")
print("Type 'exit' to stop.\n")

while True:
    query = input("You: ")

    if query.lower() == "exit":
        print("Chat ended.")
        break

    answer = ask(query)
    print("\nBot:", answer)

 Multi-PDF College Assistant Ready!
Type 'exit' to stop.

You: what is Admission Procedures

Bot: Not in documents.
You: when is last date for the admission ?

Bot: Not in documents.
You: exit
Chat ended.
